In [ ]:
import subprocess
import sys


def dump_db_structure_and_samples(host, port, user, password, db):
    """
    Dumps the structure of a MySQL database and sample rows from each table.

    Args:
        host (str): The hostname or IP address of the MySQL server.
        port (str): The port number of the MySQL server.
        user (str): The username to connect with.
        password (str): The password to connect with.
        db (str): The name of the database to dump.

    Returns:
        str: The dump content as a single string.
    """
    
    mysqldump_cmd = [
        'mysqldump',
        f'-h{host}',
        f'-P{port}',
        f'-u{user}',
        f'-p{password}',
        '--no-data',           
        '--single-transaction',
        db
    ]

    try:
        
        result = subprocess.run(
            mysqldump_cmd,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False  
        )
        
        try:
            dump_content = result.stdout.decode('utf-8')
        except UnicodeDecodeError:
            dump_content = result.stdout.decode('latin-1')
    except subprocess.CalledProcessError as e:
        
        try:
            error_msg = e.stderr.decode('utf-8')
        except UnicodeDecodeError:
            error_msg = e.stderr.decode('latin-1')
        print(f"Error running mysqldump: {error_msg}")
        return ""

    
    mysql_show_tables_cmd = [
        'mysql',
        f'-h{host}',
        f'-P{port}',
        f'-u{user}',
        f'-p{password}',
        '-D', db,
        '-N',  
        '-e', 'SHOW TABLES;'
    ]

    try:
        result = subprocess.run(
            mysql_show_tables_cmd,
            check=True,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=False  
        )
        
        try:
            tables_raw_bytes = result.stdout
            tables_raw = tables_raw_bytes.decode('utf-8').strip()
        except UnicodeDecodeError:
            tables_raw = result.stdout.decode('latin-1').strip()

        if not tables_raw:
            return dump_content
        
        
        tables = [line.strip() for line in tables_raw.split('\n') if line.strip()]
    except subprocess.CalledProcessError as e:
        
        try:
            error_msg = e.stderr.decode('utf-8')
        except UnicodeDecodeError:
            error_msg = e.stderr.decode('latin-1')
        print(f"Error getting table list: {error_msg}")
        return ""

    
    for table in tables:
        
        
        escaped_table = table.replace('`', r'\`')
        select_cmd = [
            'mysql',
            f'-h{host}',
            f'-P{port}',
            f'-u{user}',
            f'-p{password}',
            '-D', db,
            '-e', f'SELECT * FROM `{escaped_table}` ORDER BY RAND() LIMIT 6;'
        ]

        try:
            result = subprocess.run(
                select_cmd,
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
                text=False  
            )
            
            try:
                sample_data_bytes = result.stdout
                sample_data = sample_data_bytes.decode('utf-8')
            except UnicodeDecodeError:
                sample_data = result.stdout.decode('latin-1')

            if not sample_data.strip():
                 
                dump_content += f"\n-- No sample data returned for table: {table}\n"
                continue

            
            commented_sample = ''.join([f'-- {line}' for line in sample_data.splitlines(True)])

            dump_content += f"\n-- Sample data from table: {table}\n"
            dump_content += commented_sample
            dump_content += "\n"
        except subprocess.CalledProcessError as e:
            
            try:
                error_msg = e.stderr.decode('utf-8')
            except UnicodeDecodeError:
                error_msg = e.stderr.decode('latin-1')
            print(f"Error getting sample data for table {table}: {error_msg}")
            
            
            continue

    return dump_content

In [4]:
HOST = "relational.fel.cvut.cz"
PORT = "3306"
USER = "guest"
PASSWORD = "ctu-relational"  
DB_NAME = "pubs"         

output_str = dump_db_structure_and_samples(HOST, PORT, USER, PASSWORD, DB_NAME)
print(output_str)

/*M!999999\- enable the sandbox mode */ 
-- MariaDB dump 10.19  Distrib 10.11.13-MariaDB, for debian-linux-gnu (x86_64)
--
-- Host: relational.fel.cvut.cz    Database: pubs
-- ------------------------------------------------------
-- Server version	10.6.12-MariaDB-1:10.6.12+maria~ubu2004-log

/*!40101 SET @OLD_CHARACTER_SET_CLIENT=@@CHARACTER_SET_CLIENT */;
/*!40101 SET @OLD_CHARACTER_SET_RESULTS=@@CHARACTER_SET_RESULTS */;
/*!40101 SET @OLD_COLLATION_CONNECTION=@@COLLATION_CONNECTION */;
/*!40101 SET NAMES utf8mb4 */;
/*!40103 SET @OLD_TIME_ZONE=@@TIME_ZONE */;
/*!40103 SET TIME_ZONE='+00:00' */;
/*!40014 SET @OLD_UNIQUE_CHECKS=@@UNIQUE_CHECKS, UNIQUE_CHECKS=0 */;
/*!40014 SET @OLD_FOREIGN_KEY_CHECKS=@@FOREIGN_KEY_CHECKS, FOREIGN_KEY_CHECKS=0 */;
/*!40101 SET @OLD_SQL_MODE=@@SQL_MODE, SQL_MODE='NO_AUTO_VALUE_ON_ZERO' */;
/*!40111 SET @OLD_SQL_NOTES=@@SQL_NOTES, SQL_NOTES=0 */;

--
-- Table structure for table `authors`
--

DROP TABLE IF EXISTS `authors`;
/*!40101 SET @saved_cs_client 

In [39]:
def get_sql_agg_prompt(description):
    return f"""
You're an SQL expert. Below is a description of the database and sample data from its tables.

Task:  
Create one SQL query that represents a **multi-hop analysis** - meaning the answer requires traversing relationships across multiple tables to compute the result. The query must:

- Perform a multi-hop JOIN across at least three tables (e.g., Table A → Table B → Table C) where each join reveals necessary information for the final calculation
- Use exactly one aggregate function to compute a single scalar value that depends on data from all joined tables
- Include at least one WHERE or HAVING condition that filters data based on criteria requiring information from different tables
- Ensure the result is exactly one row with one value, representing a meaningful business metric that requires this multi-table traversal

The query should answer a complex analytical question that cannot be answered by looking at just one or two tables - it must require the full multi-table path to compute the correct result.

Also provide a second version of the same query that:  
- Uses SELECT * to show all intermediate data
- Keeps the exact same multi-hop JOINs and table relationships
- Removes GROUP BY and the aggregate function
- Retains all WHERE conditions from the first query to show the filtered dataset before aggregation

Both queries must be valid for MySQL MariaDB. Return the result **only** in the following JSON format with no additional text:

{{
    "aggregated_query": "FIRST_QUERY_HERE",
    "select_star_query": "SECOND_QUERY_HERE"
}}

DB DESCRIPTION:
{description}

"""

In [40]:
def get_sql_select_prompt(description):
    return f"""
You're an SQL expert. Below is a description of the database and sample data from its tables.

Task:  
Create one SQL query that:  

- Joins at least three tables.  
- Returns exactly **one row with one single scalar value** (i.e., the SELECT clause must contain only one column or expression that yields a single result).  
- Includes a WHERE condition to ensure the result is uniquely determined (e.g., by ID, unique combination, or using LIMIT 1 if appropriate).  
- Must not use aggregate functions unless necessary to guarantee a single value—but the primary goal is to fetch one specific value, not to compute a summary.  

Also provide a second version of the same query that:  

- Uses `SELECT *`  
- Keeps the exact same JOINs and WHERE conditions  
- Removes any `LIMIT`, `DISTINCT`, or other clauses used solely to enforce a single result in the first query, **unless** they are logically part of the filtering (e.g., keep WHERE but drop LIMIT 1 if it was only for single-row guarantee)  

Both queries must be valid for MySQL MariaDB. Return the result **only** in the following JSON format with no additional text:

{{
    "aggregated_query": "FIRST_QUERY_HERE",
    "select_star_query": "SECOND_QUERY_HERE"
}}

DATABASE DESCRIPTION:  

{description}
"""

In [ ]:
import requests
import os
import json

def query_openrouter(
    prompt: str,
    model: str = "meta-llama/llama-3.1-8b-instruct:free",
    api_key: str | None = None,
    temperature: float = 0.7,
    max_tokens: int = 512,
    site_url: str | None = None,
    site_title: str | None = None,
):
    """
    Отправляет запрос к OpenRouter API и возвращает сгенерированный ответ.

    :param prompt: Текст запроса (prompt).
    :param model: Название модели (например, "meta-llama/llama-3.1-8b-instruct:free").
    :param api_key: Ключ API OpenRouter. Если не указан — берётся из переменной окружения OPENROUTER_API_KEY.
    :param temperature: Температура генерации (по умолчанию 0.7).
    :param max_tokens: Максимальное количество токенов в ответе (по умолчанию 512).
    :param site_url: Опционально. URL вашего сайта для рейтингов на openrouter.ai.
    :param site_title: Опционально. Название вашего сайта для рейтингов на openrouter.ai.
    :return: Строка с ответом модели или None в случае ошибки.
    """
    if api_key is None:
        api_key = os.getenv("OPENROUTER_API_KEY")
        if not api_key:
            raise ValueError("Не указан API-ключ. Установите переменную окружения OPENROUTER_API_KEY или передайте ключ напрямую.")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    
    if site_url is not None:
        headers["HTTP-Referer"] = site_url
    if site_title is not None:
        headers["X-Title"] = site_title

    payload = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }

    try:
        response = requests.post(
            url="https://openrouter.ai/api/v1/chat/completions",
            headers=headers,
            data=json.dumps(payload)
        )
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"].strip()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при обращении к OpenRouter API: {e}")
        return None

In [42]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import re

def split_sql_queries(text: str) -> list[str]:
    """
    Разделяет текст, содержащий один или несколько SQL-запросов,
    на отдельные валидные SQL-запросы без лишних пробелов и пустых строк.

    Поддерживает разделение как по ';' так и по двойным переносам строк.

    :param text: Строка с одним или несколькими SQL-запросами.
    :return: Список строк, каждый элемент — корректный SQL-запрос.
    """
    
    
    parts = re.split(r';\s*', text.strip())
    
    queries = []
    for part in parts:
        
        cleaned = part.strip()
        if cleaned:  
            queries.append(cleaned)

    
    if len(queries) == 1 and '\n\n' in queries[0]:
        candidates = re.split(r'\n\s*\n', queries[0])
        alt_queries = [q.strip() for q in candidates if q.strip()]
        if len(alt_queries) > 1:
            queries = alt_queries

    return queries

In [ ]:
import re
import sqlite3
import pymysql
import datetime
from typing import Dict, List, Set, Any, Tuple, Optional
from collections import defaultdict
import random




MYSQL_METADATA_QUERIES = {
    'get_all_tables': """
        SELECT TABLE_NAME 
        FROM INFORMATION_SCHEMA.TABLES 
        WHERE TABLE_SCHEMA = %s
    """,
    
    'get_table_columns': """
        SELECT 
            COLUMN_NAME, 
            DATA_TYPE, 
            IS_NULLABLE, 
            COLUMN_KEY,
            EXTRA
        FROM INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = %s AND TABLE_NAME = %s
        ORDER BY ORDINAL_POSITION
    """,
    
    'get_primary_keys': """
        SELECT COLUMN_NAME
        FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
        WHERE TABLE_SCHEMA = %s 
          AND TABLE_NAME = %s
          AND CONSTRAINT_NAME = 'PRIMARY'
        ORDER BY ORDINAL_POSITION
    """
}



def _convert_value_for_sqlite(value):
    """Преобразует значение в тип, поддерживаемый SQLite."""
    if value is None:
        return None
    
    
    if isinstance(value, datetime.timedelta):
        total_seconds = int(value.total_seconds())
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        seconds = total_seconds % 60
        microseconds = value.microseconds
        if microseconds:
            return f"{hours:02d}:{minutes:02d}:{seconds:02d}.{microseconds:06d}"
        return f"{hours:02d}:{minutes:02d}:{seconds:02d}"
    
    if isinstance(value, datetime.time):
        return value.strftime("%H:%M:%S.%f") if value.microsecond else value.strftime("%H:%M:%S")
    
    if isinstance(value, (datetime.date, datetime.datetime)):
        return value.isoformat()
    
    
    if isinstance(value, bytes):
        try:
            return value.decode('utf-8', errors='replace')
        except:
            return str(value)
    
    
    if isinstance(value, (int, float, str, bool)):
        return value
    
    return str(value)

def _find_matching_table(name: str, all_tables: List[str]) -> Optional[str]:
    """Находит таблицу с совпадающим именем без учета регистра."""
    for table in all_tables:
        if table.lower() == name.lower():
            return table
    return None

def _normalize_query_for_db(query: str, all_tables: List[str], db_type: str = 'mysql') -> str:
    """Нормализует SQL-запрос для указанной БД."""
    
    query = re.sub(r'--.*?$|/\*.*?\*/', '', query, flags=re.DOTALL | re.MULTILINE)
    
    
    if db_type == 'mysql':
        for table in all_tables:
            pattern = r'\b' + re.escape(table) + r'\b'
            query = re.sub(pattern, f'`{table}`', query, flags=re.IGNORECASE)
    
    
    elif db_type == 'sqlite':
        query = query.replace('`', '"')
        query = re.sub(r'"\w+"\."(\w+)"', r'"\1"', query)
        
        for table in all_tables:
            pattern = r'\b' + re.escape(table) + r'\b'
            query = re.sub(pattern, f'"{table.lower()}"', query, flags=re.IGNORECASE)
        
        
        query = re.sub(r'NOW\(\)', 'CURRENT_TIMESTAMP', query, flags=re.IGNORECASE)
        query = re.sub(r'CURDATE\(\)', 'CURRENT_DATE', query, flags=re.IGNORECASE)
        query = re.sub(r'CURTIME\(\)', 'CURRENT_TIME', query, flags=re.IGNORECASE)
        query = re.sub(r'RAND\(\)', 'RANDOM()', query, flags=re.IGNORECASE)
    
    return query.strip()

def _extract_tables_from_query(query: str, all_tables: List[str]) -> List[str]:
    """Извлекает имена таблиц из SQL-запроса с точным сопоставлением."""
    potential_tables = set()
    
    
    table_pattern = r'(?:FROM|JOIN)\s+([`"\w.]+)(?:\s+(?:AS\s+)?\w+)?'
    matches = re.findall(table_pattern, query, re.IGNORECASE)
    
    for match in matches:
        clean_name = match.strip('`"').split('.')[-1]
        potential_tables.add(clean_name)
    
    
    subquery_pattern = r'\(\s*SELECT.*?FROM\s+([`"\w.]+)'
    matches = re.findall(subquery_pattern, query, re.IGNORECASE | re.DOTALL)
    for match in matches:
        clean_name = match.strip('`"').split('.')[-1]
        potential_tables.add(clean_name)
    
    
    found_tables = set()
    for potential in potential_tables:
        matched_table = _find_matching_table(potential, all_tables)
        if matched_table:
            found_tables.add(matched_table)
    
    return list(found_tables)



def get_all_table_names(cursor, metadata_queries: Dict[str, str], database: str) -> List[str]:
    """Получает список всех таблиц в базе данных."""
    cursor.execute(metadata_queries['get_all_tables'], (database,))
    return [row['TABLE_NAME'] for row in cursor.fetchall()]

def get_table_schemas(cursor, table_names: List[str], all_tables: List[str], 
                     metadata_queries: Dict[str, str], database: str) -> Dict[str, Dict]:
    """Получает схемы таблиц."""
    schemas = {}
    
    for table in table_names:
        if not table:
            continue
            
        if table not in all_tables:
            matched_table = _find_matching_table(table, all_tables)
            if matched_table:
                table = matched_table
            else:
                continue
        
        
        cursor.execute(metadata_queries['get_table_columns'], (database, table))
        columns = cursor.fetchall()
        
        if not columns:
            continue
        
        
        cursor.execute(metadata_queries['get_primary_keys'], (database, table))
        pk_columns = {row['COLUMN_NAME'] for row in cursor.fetchall()}
        
        schemas[table] = {
            'columns': columns,
            'primary_keys': pk_columns
        }
    
    return schemas



def execute_query(cursor, query: str, all_tables: List[str]) -> List[Dict]:
    """Выполняет SQL-запрос и возвращает результаты."""
    normalized_query = _normalize_query_for_db(query, all_tables, 'mysql')
    cursor.execute(normalized_query)
    return cursor.fetchall()

def fetch_table_data(cursor, table_name: str, schema: Dict, 
                    pk_values: Set, actual_row_limit: int) -> List[Dict]:
    """Получает данные таблицы с приоритетом строк по PK и дополнением до лимита случайными строками."""
    pk_columns = list(schema['primary_keys'])
    columns = [f"`{col['COLUMN_NAME']}`" for col in schema['columns']]
    
    rows = []
    
    if pk_values and pk_columns:
        pk_col = pk_columns[0]
        format_str = ','.join(['%s'] * len(pk_values))
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            WHERE `{pk_col}` IN ({format_str})
            LIMIT {actual_row_limit}
        """
        try:
            cursor.execute(query, tuple(pk_values))
            rows = cursor.fetchall()
        except (pymysql.Error, KeyError):
            pass
    
    
    if len(rows) < actual_row_limit:
        additional_limit = actual_row_limit - len(rows)
        where_clause = ""
        params = []
        
        
        if pk_columns and rows:
            pk_col = pk_columns[0]
            exclude_ids = [
                row[pk_col] for row in rows 
                if pk_col in row and row[pk_col] is not None
            ]
            if exclude_ids:
                format_str = ','.join(['%s'] * len(exclude_ids))
                where_clause = f"WHERE `{pk_col}` NOT IN ({format_str})"
                params = exclude_ids
        
        
        try:
            query = f"""
                SELECT {', '.join(columns)}
                FROM `{table_name}`
                {where_clause}
                ORDER BY RAND()  -- Получаем случайные строки
                LIMIT {additional_limit}
            """
            cursor.execute(query, tuple(params))
            additional_rows = cursor.fetchall()
            rows.extend(additional_rows)
        except pymysql.Error:
            
            query = f"""
                SELECT {', '.join(columns)}
                FROM `{table_name}`
                {where_clause}
                LIMIT {additional_limit}
            """
            cursor.execute(query, tuple(params))
            additional_rows = cursor.fetchall()
            rows.extend(additional_rows)
    
    
    return rows[:actual_row_limit]

def fetch_random_table_data(cursor, table_name: str, schema: Dict, 
                           actual_row_limit: int) -> List[Dict]:
    """Получает случайные строки из таблицы."""
    columns = [f"`{col['COLUMN_NAME']}`" for col in schema['columns']]
    
    try:
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            ORDER BY RAND()
            LIMIT {actual_row_limit}
        """
        cursor.execute(query)
        return cursor.fetchall()
    except pymysql.Error:
        
        query = f"""
            SELECT {', '.join(columns)}
            FROM `{table_name}`
            LIMIT {actual_row_limit}
        """
        cursor.execute(query)
        return cursor.fetchall()

def collect_pk_values(detail_rows: List[Dict], table_schemas: Dict[str, Dict]) -> Dict[str, Set]:
    """Собирает значения первичных ключей из результата запроса."""
    pk_values_map = defaultdict(set)
    
    for row in detail_rows:
        for table_name, schema in table_schemas.items():
            pk_columns = schema['primary_keys']
            if not pk_columns:
                continue
            
            table_prefix = table_name.lower()
            
            for pk_col in pk_columns:
                candidates = [
                    pk_col,
                    f"{table_prefix}.{pk_col}",
                    f"{table_name}.{pk_col}",
                    pk_col.lower(),
                    f"{table_prefix}.{pk_col.lower()}"
                ]
                
                for candidate in candidates:
                    if candidate in row and row[candidate] is not None:
                        pk_values_map[table_name].add(row[candidate])
                        break
    
    return pk_values_map



def create_sqlite_database(db_path: str, schemas: Dict[str, Dict], 
                          table_data: Dict[str, List[Dict]]) -> None:
    """Создает SQLite-базу и наполняет данными."""
    type_mapping = {
        'int': 'INTEGER',
        'tinyint': 'INTEGER',
        'smallint': 'INTEGER',
        'mediumint': 'INTEGER',
        'bigint': 'INTEGER',
        'float': 'REAL',
        'double': 'REAL',
        'decimal': 'REAL',
        'numeric': 'REAL',
        'date': 'TEXT',
        'datetime': 'TEXT',
        'timestamp': 'TEXT',
        'time': 'TEXT',
        'year': 'INTEGER',
        'char': 'TEXT',
        'varchar': 'TEXT',
        'text': 'TEXT',
        'blob': 'BLOB',
        'enum': 'TEXT',
        'set': 'TEXT'
    }
    
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        cursor.execute("PRAGMA foreign_keys = ON;")
        
        sqlite_table_names = {}
        
        for original_table_name, schema in schemas.items():
            if not schema['columns']:
                continue
                
            sqlite_table_name = original_table_name.lower()
            sqlite_table_names[original_table_name] = sqlite_table_name
            
            columns_def = []
            pk_constraints = []
            
            for col in schema['columns']:
                col_name = col['COLUMN_NAME']
                mysql_type = col['DATA_TYPE'].lower()
                sqlite_type = 'TEXT'
                
                for key, value in type_mapping.items():
                    if key in mysql_type:
                        sqlite_type = value
                        break
                
                is_auto_increment = 'auto_increment' in col['EXTRA'].lower()
                nullable = '' if col['IS_NULLABLE'] == 'YES' else 'NOT NULL'
                
                if is_auto_increment and 'INTEGER' in sqlite_type:
                    column_def = f'"{col_name.lower()}" {sqlite_type} PRIMARY KEY AUTOINCREMENT'
                else:
                    column_def = f'"{col_name.lower()}" {sqlite_type} {nullable}'
                    if col_name in schema['primary_keys']:
                        pk_constraints.append(f'"{col_name.lower()}"')
                
                columns_def.append(column_def)
            
            if pk_constraints:
                pk_str = ", ".join(pk_constraints)
                columns_def.append(f"PRIMARY KEY ({pk_str})")
            
            create_query = f'CREATE TABLE IF NOT EXISTS "{sqlite_table_name}" ({", ".join(columns_def)})'
            try:
                cursor.execute(create_query)
            except sqlite3.OperationalError:
                continue
        
        
        for original_table_name, rows in table_data.items():
            if not rows or original_table_name not in sqlite_table_names:
                continue
                
            sqlite_table_name = sqlite_table_names[original_table_name]
            columns = list(rows[0].keys())
            columns_lower = [col.lower() for col in columns]
            
            try:
                cursor.execute(f"PRAGMA table_info(\"{sqlite_table_name}\")")
                existing_columns = {row[1].lower() for row in cursor.fetchall()}
            except sqlite3.OperationalError:
                continue
            
            valid_columns = [col for col in columns_lower if col in existing_columns]
            
            if not valid_columns:
                continue
            
            placeholders = ", ".join(["?"] * len(valid_columns))
            insert_query = f"""
                INSERT OR IGNORE INTO "{sqlite_table_name}" 
                ({', '.join(f'"{col}"' for col in valid_columns)}) 
                VALUES ({placeholders})
            """
            
            for row in rows:
                values = []
                for orig_col in columns:
                    lower_col = orig_col.lower()
                    if lower_col in existing_columns:
                        value = row.get(orig_col)
                        if value is None and lower_col in row:
                            value = row.get(lower_col)
                        values.append(_convert_value_for_sqlite(value))
                
                try:
                    cursor.execute(insert_query, values)
                except (sqlite3.IntegrityError, sqlite3.OperationalError):
                    pass
        
        conn.commit()

def validate_query_in_sqlite(db_path: str, query: str, all_tables: List[str]) -> Tuple[bool, Any]:
    """Выполняет запрос в SQLite и возвращает результат."""
    with sqlite3.connect(db_path) as conn:
        cursor = conn.cursor()
        normalized_query = _normalize_query_for_db(query, all_tables, 'sqlite')
        
        try:
            cursor.execute(normalized_query)
            result = cursor.fetchone()
            return True, result
        except sqlite3.OperationalError as e:
            return False, str(e)



def create_local_db_snapshot(
    remote_config: Dict[str, Any],
    aggregation_query: str,
    detail_query: str,
    metadata_queries: Dict[str, str] = None,
    output_db_path: str = "local_snapshot.db",
    row_limit: int = 40,
    essential_tables: List[str] = None
) -> None:
    """
    Создает локальную SQLite-копию удаленной MySQL-базы.
    
    Args:
        remote_config: Конфигурация подключения к MySQL
        aggregation_query: Агрегирующий запрос для проверки
        detail_query: Детальный запрос для получения данных
        metadata_queries: Словарь с запросами для получения метаданных
        output_db_path: Путь к выходному SQLite файлу
        row_limit: Лимит строк на таблицу
        essential_tables: Список обязательных таблиц
    """
    if metadata_queries is None:
        metadata_queries = MYSQL_METADATA_QUERIES
    
    if essential_tables is None:
        essential_tables = []
    
    
    MIN_ROWS = 20
    
    MAX_ROWS = max(MIN_ROWS, row_limit)
    
    print("\033[94m" + "="*60)
    print("Создание локального снапшота БД")
    print("="*60 + "\033[0m")
    
    
    with pymysql.connect(**remote_config) as remote_conn:
        with remote_conn.cursor(pymysql.cursors.DictCursor) as cursor:
            
            
            all_tables = get_all_table_names(
                cursor, metadata_queries, remote_config['database']
            )
            print(f"Всего таблиц в базе: {len(all_tables)}")
            
            
            dependent_tables = set()
            dependent_tables.update(_extract_tables_from_query(detail_query, all_tables))
            dependent_tables.update(_extract_tables_from_query(aggregation_query, all_tables))
            
            
            for table in essential_tables:
                matched_table = _find_matching_table(table, all_tables)
                if matched_table:
                    dependent_tables.add(matched_table)
            
            dependent_tables = list(dependent_tables)
            print(f"Таблицы для снапшота: {len(dependent_tables)}")
            
            
            
            
            
            table_schemas = get_table_schemas(
                cursor, dependent_tables, all_tables, 
                metadata_queries, remote_config['database']
            )
            
            
            print("\nВыполнение детального запроса...")
            detail_rows = execute_query(cursor, detail_query, all_tables)
            print(f"Получено {len(detail_rows)} строк")
            
            if not detail_rows:
                raise ValueError("Детальный запрос вернул пустой результат")
            
            
            pk_values_map = collect_pk_values(detail_rows, table_schemas)
            
            
            table_data = {}
            for table_name in dependent_tables:
                if table_name in table_schemas:
                    
                    actual_row_limit = random.randint(MIN_ROWS, MAX_ROWS)
                    table_data[table_name] = fetch_table_data(
                        cursor, table_name, table_schemas[table_name],
                        pk_values_map.get(table_name, set()), actual_row_limit
                    )
                    print(f"Загружено {len(table_data[table_name])} строк из {table_name} (лимит: {actual_row_limit})")
            
            
            remaining_tables = [t for t in all_tables if t not in dependent_tables]
            
            if remaining_tables:
                remaining_schemas = get_table_schemas(
                    cursor, remaining_tables, all_tables,
                    metadata_queries, remote_config['database']
                )
                table_schemas.update(remaining_schemas)
                
                for table_name in remaining_tables:
                    if table_name in remaining_schemas:
                        
                        actual_row_limit = random.randint(MIN_ROWS, MAX_ROWS)
                        table_data[table_name] = fetch_random_table_data(
                            cursor, table_name, remaining_schemas[table_name], actual_row_limit
                        )
                        print(f"Загружено {len(table_data[table_name])} строк из {table_name} (лимит: {actual_row_limit})")
    
    
    print(f"\nСоздание локальной базы: {output_db_path}")
    create_sqlite_database(output_db_path, table_schemas, table_data)
    
    
    print("\nПроверка агрегирующего запроса в SQLite...")
    success, result = validate_query_in_sqlite(
        output_db_path, aggregation_query, all_tables
    )
    
    if success:
        print(f"\033[92mЗапрос успешно выполнен. Результат: {result}\033[0m")
    else:
        print(f"\033[93mОшибка выполнения запроса: {result}\033[0m")
    
    print(f"\n\033[92mСнапшот сохранен в {output_db_path}\033[0m")

    return result

In [ ]:
import sqlite3
import pandas as pd

def execute_sql_query(db_file, query):
    """
    Выполняет SQL-запрос к локальной базе данных и возвращает результат в виде DataFrame.

    :param db_file: путь к файлу локальной базы данных SQLite
    :param query: SQL-запрос для выполнения
    :return: результат запроса в виде pandas DataFrame
    """
    
    conn = sqlite3.connect(db_file)
    
    try:
        
        result_df = pd.read_sql_query(query, conn)
        return result_df
    except Exception as e:
        print(f"Произошла ошибка при выполнении запроса: {e}")
        return None
    finally:
        
        conn.close()

In [18]:
query = """
SELECT DISTINCT o.OrderID AS TotalOrders FROM Orders o JOIN Customers c ON o.CustomerID = c.CustomerID JOIN Employees e ON o.EmployeeID = e.EmployeeID WHERE c.Country = 'USA' AND e.City = 'Seattle';
"""

In [19]:
db_file = f'/home/george/Documents/code/vkr/rustbench/tables-qa/set/northwind/3/local_snapshot.db'
result = execute_sql_query(db_file, query)
print(result)

    TotalOrders
0         10262
1         10305
2         10310
3         10314
4         10316
5         10369
6         10385
7         10393
8         10394
9         10401
10        10452
11        10469
12        10482
13        10545
14        10579
15        10589
16        10596
17        10598
18        10603
19        10612
20        10616
21        10627
22        10660
23        10665
24        10680
25        10696
26        10706
27        10713
28        10719
29        10722
30        10756
31        10821
32        10852
33        10883
34        10894
35        10984
36        10992
37        11034
38        11064
39        11077


In [ ]:
import requests
import json

def query_ollama(
    prompt: str,
    model: str = "qwen3:8b",
    host: str = "localhost",
    port: int = 11434,
    timeout: int = 120,
    think: bool = None
) -> str | None:
    """
    Отправляет prompt локальной модели через Ollama API и возвращает сгенерированный текст.

    :param prompt: Текст запроса.
    :param model: Название модели, например "llama3.2", "mistral", "phi3" и т.д.
    :param host: Хост, на котором запущен Ollama (по умолчанию "localhost").
    :param port: Порт Ollama API (по умолчанию 11434).
    :param timeout: Таймаут ожидания ответа в секундах.
    :return: Строка с ответом модели или None в случае ошибки.
    """
    url = f"http://{host}:{port}/api/generate"
    
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False  
    }

    if think:
        payload["think"] = think

    headers = {"Content-Type": "application/json"}

    try:
        response = requests.post(url, json=payload, headers=headers, timeout=timeout)
        response.raise_for_status()
        result = response.json()
        return result.get("response", "").strip()
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при обращении к Ollama API: {e}")
        return None

In [50]:
def get_sql_to_text_prompt(query):
    return f"""
    Преобразуй следующий SQL-запрос в вопрос на естественном русском языке, 
    как если бы его задал человек, не знакомый с базами данных или SQL. 
    Вопрос должен быть понятен обычному пользователю, не должен содержать терминов вроде «таблица», 
    «JOIN» или «запрос», и должен точно отражать смысл операций в SQL (фильтрацию, объединение данных, 
    агрегацию и т.д.) в повседневной формулировке. Формулируй его либо как вопросительную конструкцию 
    (начиная со слов "Как", "Сколько" и т.д.), либо как поручение ("Выведи", "Найди" и т.д.)

    Верни только сформулированный вопрос, без дополнительного текста.
    SQL-запрос:  
    {query}
    """

In [244]:
from pprint import pprint

In [ ]:
import os
import requests
from typing import Optional, Any, Dict

def query_codestral_chat(
    prompt: str,
    model: str = "codestral-latest",  
    api_key: Optional[str] = None,
    return_json: bool = True,
    temperature: float = 0.7,
    max_tokens: Optional[int] = None,
) -> Optional[str]:
    """
    Отправляет запрос к Codestral через REST API напрямую.

    :param prompt: Текст запроса (сообщение от пользователя).
    :param model: Идентификатор модели (по умолчанию "codestral-latest").
    :param api_key: API-ключ Mistral. Если не указан — берётся из MISTRAL_API_KEY.
    :param return_json: Если True, добавляется response_format={"type": "json_object"}.
    :param temperature: Температура генерации (по умолчанию 0.7).
    :param max_tokens: Максимальное число токенов в ответе (опционально).
    :return: Текст ответа модели или None в случае ошибки.
    """
    if api_key is None:
        api_key = os.getenv("MISTRAL_API_KEY")
        if not api_key:
            raise ValueError("Не задан API-ключ. Установите переменную окружения MISTRAL_API_KEY.")

    url = "https://codestral.mistral.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    payload: Dict[str, Any] = {
        "model": model,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
    }

    if max_tokens is not None:
        payload["max_tokens"] = max_tokens

    if return_json:
        payload["response_format"] = {"type": "json_object"}

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        data = response.json()
        return data["choices"][0]["message"]["content"]
    except requests.exceptions.RequestException as e:
        print(f"Ошибка при вызове Codestral API: {e}")
        if response is not None:
            print("Ответ сервера:", response.text)
        return None

In [85]:
load_dotenv()
mistral_key = os.getenv("MISTRAL_API_KEY")
codestral_key = os.getenv("CODESTRAL_KEY")

In [ ]:


def pipeline(db_name, number, sql=None, mode=None):
    HOST = "relational.fel.cvut.cz"
    PORT = "3306"
    USER = "guest"
    PASSWORD = "ctu-relational"  
    DB_NAME = db_name      
    path = f"set/{db_name}/{number}"

    os.makedirs(path, exist_ok=True)
    
    if not sql:
        output_str = dump_db_structure_and_samples(HOST, PORT, USER, PASSWORD, DB_NAME)

        if mode=="select":
            result = query_codestral_chat(get_sql_select_prompt(output_str), api_key=codestral_key, return_json=True, temperature=0.85)
        else:
            result = query_codestral_chat(get_sql_agg_prompt(output_str), api_key=codestral_key, return_json=True, temperature=0.85)
        
        
        print(result)

        queries = json.loads(result)
    else:
        queries = sql

    remote_config = {
    'host': 'relational.fel.cvut.cz',
    'port': 3306,
    'user': 'guest',
    'password': 'ctu-relational',
    'database': db_name,
    'charset': 'utf8mb4',
    'cursorclass': pymysql.cursors.DictCursor
    }

    
    aggregation_query = queries["aggregated_query"]
    detail_query = queries["select_star_query"]

    is_empty = True

    while is_empty:
    
        query_result = create_local_db_snapshot(
            remote_config=remote_config,
            aggregation_query=aggregation_query,
            detail_query=detail_query,
            output_db_path=f"{path}/local_snapshot.db",
            row_limit=90,
        )
        

        if "None" not in str(query_result[0]) and query_result[0] != 0:
            is_empty = False

    natural_question = query_ollama(get_sql_to_text_prompt(aggregation_query), model="qwen3:8b", think=False)

    QA = f"Q:{natural_question}\nA:{query_result}"
    pprint(QA)

    with open(f"{path}/QA.txt", "w") as f:
        f.write(QA)
    
    with open(f"{path}/aggregation_query.sql", "w") as f:
        f.write(aggregation_query)
    with open(f"{path}/detail_query.sql", "w") as f:
        f.write(detail_query)

In [55]:
from tqdm import tqdm

In [56]:
ready_name = []

In [57]:
ready_name

[]

In [70]:
for name in os.listdir("set"):
    for number in os.listdir(f"set/{name}"):
        if len(os.listdir(f"set/{name}/{number}")) < 4:
            print(name, number)

lahman_2014 3
ErgastF1 2


In [58]:
for name in tqdm(os.listdir("set")):
    if name in ready_name:
        continue

    while max([int(i) for i in os.listdir(f"set/{name}") if len(os.listdir(f"set/{name}/{i}")) >= 4]) != 10:
        try:
            for number in range(max([int(i) for i in os.listdir(f"set/{name}") if len(os.listdir(f"set/{name}/{i}")) >= 4]) + 1, 11):
                print(name, number)
                pipeline(name, number)
        except Exception:
            print("Exception")
    
    ready_name.append(name)

100%|██████████| 10/10 [00:00<00:00, 4766.25it/s]


In [83]:
from pprint import pprint

In [ ]:
name = "northwind"
number = 10
sql = {
    "aggregated_query": "SELECT COUNT(*) AS order_count FROM (SELECT DISTINCT o.OrderID FROM Orders o JOIN `Order Details` od ON o.OrderID = od.OrderID JOIN Products p ON od.ProductID = p.ProductID WHERE p.CategoryID = 8) AS distinct_orders;",
    "select_star_query": """SELECT o.OrderID, o.CustomerID, o.EmployeeID, o.OrderDate, o.RequiredDate, o.ShippedDate, o.ShipVia, o.Freight, o.ShipName, o.ShipAddress, o.ShipCity, o.ShipRegion, o.ShipPostalCode, o.ShipCountry, od.ProductID, p.ProductName, p.CategoryID, p.CategoryName 
FROM Orders o 
JOIN `Order Details` od ON o.OrderID = od.OrderID 
JOIN Products p ON od.ProductID = p.ProductID 
WHERE p.CategoryID = 8;"""
}

while len(os.listdir(f"set/{name}/{number}")) != 4:
    pipeline(name, number)
    
    
    
    
    
    


{
    "aggregated_query": "SELECT SUM(od.Quantity * od.UnitPrice) AS TotalSales FROM Orders o JOIN Order Details od ON o.OrderID = od.OrderID JOIN Products p ON od.ProductID = p.ProductID JOIN Categories c ON p.CategoryID = c.CategoryID WHERE o.OrderDate BETWEEN '1997-01-01' AND '1997-12-31' AND c.CategoryName = 'Seafood';",
    "select_star_query": "SELECT * FROM Orders o JOIN Order Details od ON o.OrderID = od.OrderID JOIN Products p ON od.ProductID = p.ProductID JOIN Categories c ON p.CategoryID = c.CategoryID WHERE o.OrderDate BETWEEN '1997-01-01' AND '1997-12-31' AND c.CategoryName = 'Seafood';"
}
Создание локального снапшота БД
Всего таблиц в базе: 29
Таблицы для снапшота: 3

Выполнение детального запроса...
Получено 162 строк
Загружено 34 строк из Orders (лимит: 34)
Загружено 25 строк из Products (лимит: 25)
Загружено 8 строк из Categories (лимит: 28)
Загружено 1 строк из Category Sales for 1997 (лимит: 29)
Загружено 1 строк из Invoices (лимит: 78)
Загружено 1 строк из Customer

In [ ]:
import pandas as pd
from pathlib import Path
from collections import defaultdict
import numpy as np

def analyze_csv_statistics_pathlib(root_folder):
    """
    Анализирует статистику CSV таблиц в папках (версия с Pathlib)
    """
    root_path = Path(root_folder)
    
    if not root_path.exists() or not root_path.is_dir():
        print(f"Ошибка: Папка '{root_folder}' не существует!")
        return None
    
    folder_stats = []
    file_size_groups = defaultdict(list)
    
    
    for folder_path in root_path.iterdir():
        if not folder_path.is_dir():
            continue
            
        csv_files = []
        rows_per_file = []
        
        
        for csv_file in folder_path.glob("*.csv"):
            try:
                df = pd.read_csv(csv_file, encoding='utf-8', on_bad_lines='skip')
                row_count = len(df)
                
                csv_files.append(csv_file.name)
                rows_per_file.append(row_count)
                
            except Exception as e:
                print(f"Ошибка при чтении файла {csv_file}: {e}")
                continue
        
        if not csv_files:
            continue
            
        
        groups = {'0-25': 0, '25-50': 0, '50-90': 0, '90+': 0}
        
        for row_count in rows_per_file:
            if row_count <= 25:
                groups['0-25'] += 1
            elif row_count <= 50:
                groups['25-50'] += 1
            elif row_count <= 90:
                groups['50-90'] += 1
            else:
                groups['90+'] += 1
        
        
        folder_stats.append({
            'folder_name': folder_path.name,
            'file_count': len(csv_files),
            'avg_rows': np.mean(rows_per_file) if rows_per_file else 0,
            'total_rows': sum(rows_per_file),
            'min_rows': min(rows_per_file) if rows_per_file else 0,
            'max_rows': max(rows_per_file) if rows_per_file else 0,
            'groups': groups
        })
        
        
        for group_name, count in groups.items():
            file_size_groups[group_name].append(count)
    
    if not folder_stats:
        print("В указанной папке нет подпапок с CSV файлами")
        return None
    
    
    avg_files_per_folder = np.mean([fs['file_count'] for fs in folder_stats])
    avg_rows_per_folder = np.mean([fs['avg_rows'] for fs in folder_stats])
    
    avg_per_group = {}
    for group_name, counts in file_size_groups.items():
        avg_per_group[group_name] = np.mean(counts)
    
    
    print("=" * 60)
    print(f"Проанализировано папок: {len(folder_stats)}")
    print(f"Среднее количество таблиц в папке: {avg_files_per_folder:.2f}")
    print(f"Среднее количество строк в таблице: {avg_rows_per_folder:.2f}")
    
    print("\nСреднее количество таблиц по группам строк:")
    for group_name, avg_count in avg_per_group.items():
        print(f"  {group_name}: {avg_count:.2f}")
    
    return {
        'folder_stats': folder_stats,
        'avg_files_per_folder': avg_files_per_folder,
        'avg_rows_per_file': avg_rows_per_folder,
        'avg_per_group': avg_per_group
    }

In [11]:
results = analyze_csv_statistics_pathlib("/home/george/Documents/code/vkr/rustbench/tables-qa/csv")

Проанализировано папок: 100
Среднее количество таблиц в папке: 18.85
Среднее количество строк в таблице: 235.52

Среднее количество таблиц по группам строк:
  0-25: 5.39
  25-50: 2.84
  50-90: 4.40
  90+: 6.22


In [9]:
import shutil

In [10]:
base = "/home/george/Documents/code/vkr/rustbench/tables-qa/set"
for db_name in os.listdir(base):
    for number in os.listdir(os.path.join(base, db_name)):
        shutil.copytree(os.path.join(base, db_name, number, "csv"), f"/home/george/Documents/code/vkr/rustbench/tables-qa/csv/{db_name}_{number}")